# 02 — Shelf mask and regions on the OISST grid

Bathymetry (default ETOPO 2022 60″; any lat/lon grid finer than 0.25° works) is binned into each
0.25° OISST cell: a cell is **shelf** if ≥ `SHELF_FRAC` of its bathymetry points are 0–`SHELF_DEPTH` m
deep; **slope** if not shelf and its mean ocean depth is `SHELF_DEPTH`–`SLOPE_DEPTH` m.
Regions = `cfg.REGIONS` boxes ∩ shelf (first match wins).

In [ ]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import ausenso as cfg
warnings.filterwarnings("ignore", category=RuntimeWarning)
cfg.ensure_dirs()
sst0 = xr.open_zarr(cfg.F_SST).sst.isel(time=0).load()
ocean = sst0.notnull()
ny, nx = sst0.sizes["lat"], sst0.sizes["lon"]
dlat = float(sst0.lat.diff("lat").mean()); dlon = float(sst0.lon.diff("lon").mean())
lat_edges = (float(sst0.lat[0]) - dlat / 2, float(sst0.lat[-1]) + dlat / 2)
lon_edges = (float(sst0.lon[0]) - dlon / 2, float(sst0.lon[-1]) + dlon / 2)
print("OISST cell edges:", lat_edges, lon_edges)

In [ ]:
b = xr.open_dataset(cfg.BATHY_FILE)[cfg.BATHY_VAR].squeeze(drop=True)
b = b.rename({k: v for k, v in dict(latitude="lat", longitude="lon", y="lat", x="lon").items() if k in b.dims})
if float(b.lon.min()) < 0:
    b = b.assign_coords(lon=b.lon % 360)
b = b.sortby("lat").sortby("lon").sel(lat=slice(*lat_edges), lon=slice(*lon_edges)).load()
print(f"bathy: {dict(b.sizes)}, ~{abs(float(b.lat.diff('lat').mean())) * 60:.2f}′ spacing")
assert float(b.lat.min()) - lat_edges[0] < dlat / 4 and lat_edges[1] - float(b.lat.max()) < dlat / 4, "BATHY_FILE does not cover DOMAIN (lat)"
assert float(b.lon.min()) - lon_edges[0] < dlon / 4 and lon_edges[1] - float(b.lon.max()) < dlon / 4, "BATHY_FILE does not cover DOMAIN (lon)"

In [ ]:
# bin every bathymetry point into its OISST cell (robust to resolution and grid/pixel registration)
iy = np.clip(np.floor((b.lat.values - lat_edges[0]) / dlat).astype(int), 0, ny - 1)
ix = np.clip(np.floor((b.lon.values - lon_edges[0]) / dlon).astype(int), 0, nx - 1)
cell = (iy[:, None] * nx + ix[None, :]).ravel()
z = b.values.ravel().astype(float)
wet = z < 0
def binned(w):
    return np.bincount(cell, weights=w, minlength=ny * nx).reshape(ny, nx)
n = binned(np.ones_like(z))
print(f"bathy points per OISST cell: min {n.min():.0f}, median {np.median(n):.0f}")
assert n.min() >= 4, "bathymetry too coarse for the shelf-fraction method"
nwet = binned(wet.astype(float))
da = lambda a: xr.DataArray(a, coords=dict(lat=sst0.lat, lon=sst0.lon), dims=("lat", "lon"))
agg = xr.Dataset(dict(
    frac_shelf=da(binned((wet & (z >= -cfg.SHELF_DEPTH)).astype(float)) / n),
    frac_wet=da(nwet / n),
    mean_depth=da(np.where(nwet > 0, binned(np.where(wet, -z, 0.0)) / np.maximum(nwet, 1), np.nan)),
))

In [ ]:
shelf = ocean & (agg.frac_shelf >= cfg.SHELF_FRAC)
slope = ocean & ~shelf & (agg.mean_depth > cfg.SHELF_DEPTH) & (agg.mean_depth <= cfg.SLOPE_DEPTH)
names = list(cfg.REGIONS)
region = xr.full_like(sst0, -1, dtype=int)
region_slope = xr.full_like(sst0, -1, dtype=int)
for i, nm in enumerate(names):
    bx = cfg.REGIONS[nm]
    inbox = ((sst0.lat >= bx["lat"][0]) & (sst0.lat <= bx["lat"][1]) &
             (sst0.lon >= bx["lon"][0]) & (sst0.lon <= bx["lon"][1]))
    region = region.where(~(inbox & shelf & (region < 0)), i)
    region_slope = region_slope.where(~(inbox & slope & (region_slope < 0)), i)
masks = xr.Dataset(dict(ocean=ocean, shelf=shelf, slope=slope, region=region,
                        region_slope=region_slope, **agg.data_vars)).drop_vars("time", errors="ignore")
masks.attrs.update(region_names=";".join(names), shelf_depth=cfg.SHELF_DEPTH,
                   shelf_frac=cfg.SHELF_FRAC, slope_depth=cfg.SLOPE_DEPTH, bathy=str(cfg.BATHY_FILE))
masks.to_netcdf(cfg.F_MASKS)
counts = pd.Series({nm: int((region == i).sum()) for i, nm in enumerate(names)}, name="shelf cells")
counts["(shelf, unassigned)"] = int((shelf & (region < 0)).sum())
counts.to_frame()

In [ ]:
import cartopy.crs as ccrs
from matplotlib.colors import ListedColormap
fig, axs = cfg.map_axes(1, 2, figsize=(12, 5), ocean=ocean)
agg.frac_shelf.where(ocean).plot(ax=axs[0, 0], transform=ccrs.PlateCarree(), cmap="viridis",
                                 cbar_kwargs=dict(shrink=0.6, label="shelf fraction"))
axs[0, 0].contour(sst0.lon, sst0.lat, shelf.astype(float), [0.5], colors="r", linewidths=0.6,
                  transform=ccrs.PlateCarree())
cm = ListedColormap(plt.cm.tab10.colors[:len(names)])
region.where(region >= 0).plot(ax=axs[0, 1], transform=ccrs.PlateCarree(), cmap=cm,
                               vmin=-0.5, vmax=len(names) - 0.5, add_colorbar=False)
slope.where(slope).plot(ax=axs[0, 1], transform=ccrs.PlateCarree(), cmap=ListedColormap(["0.8"]),
                        add_colorbar=False)
for i, nm in enumerate(names):
    yy, xx = np.where(region.values == i)
    if len(yy):
        axs[0, 1].text(float(sst0.lon[int(np.median(xx))]), float(sst0.lat[int(np.median(yy))]), nm,
                       fontsize=7, transform=ccrs.PlateCarree(), ha="center",
                       bbox=dict(fc="w", alpha=0.7, lw=0))
axs[0, 0].set_title(f"Fraction of cell 0–{cfg.SHELF_DEPTH:.0f} m (red = shelf mask)"); axs[0, 1].set_title("Regions (grey = slope band)")
fig.savefig(cfg.FIG / "02_masks.png", dpi=200, bbox_inches="tight")